# BOJ Swap Model Practicality Analysis
モデルの収益性、リスク、および信頼性を多角的に検証し、実運用への適応可能性を評価します。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.append(os.getcwd())

from src.processing import load_and_clean_data
from src.pooling import pool_boj_data

sns.set_theme(style='whitegrid')
plt.rcParams['font.family'] = 'Hiragino Sans'
print(f"Project Root: {os.getcwd()}")

## 1. モデル予測の準備

In [ ]:
N_DAYS = 5
D_VAL = 0.4
df_cleaned = load_and_clean_data('data/BOJ_data.xlsx', 'data/BOJ_meeting_history.csv')
mpm_dates = pd.to_datetime(pd.read_csv('data/BOJ_meeting_history.csv')['Date'])
df_pooled = pool_boj_data(df_cleaned, mpm_dates=mpm_dates, d=D_VAL, n_days=N_DAYS)

features = [
    'Swap_Rate_FracDiff', 'JGB_Future_FracDiff', 'USDJPY_FracDiff', 'Nikkei225_FracDiff', 'JPY_Effective_FracDiff',
    'Spread_M3_M1_FracDiff', 'Spread_M5_M1_FracDiff', 'Spread_M8_M5_FracDiff',
    'Days_to_Next_MPM', 'Meeting_Index', 'Is_Imputed', 'Consecutive_Imputed_Days'
]
target = 'Target_FracDiff_N_Day'
df_final = df_pooled.dropna(subset=[target, 'FracDiff_Memory_Component_N'] + features)

split_date = sorted(df_final['日付'].unique())[int(len(df_final['日付'].unique()) * 0.8)]
train_df = df_final[df_final['日付'] < split_date]
test_df = df_final[df_final['日付'] >= split_date].copy()

model = CatBoostRegressor(iterations=1000, learning_rate=0.03, depth=6, verbose=0, random_seed=42)
model.fit(train_df[features], train_df[target])

test_df['Pred_Rate'] = model.predict(test_df[features]) - test_df['FracDiff_Memory_Component_N']
test_df['Actual_Move'] = test_df['Target_N_Day'] - test_df['Swap_Rate']
test_df['Pred_Move'] = test_df['Pred_Rate'] - test_df['Swap_Rate']

## 2. P&L (損益) シミュレーション
予測方向に従ってポジションを持ち、n日後の変動幅を収益とするシミュレーションです。

In [ ]:
test_df['P_L'] = np.sign(test_df['Pred_Move']) * test_df['Actual_Move']

plt.figure(figsize=(15, 8))
for idx in [1, 3, 5, 8]:
    subset = test_df[test_df['Meeting_Index'] == idx].sort_values('日付')
    plt.plot(subset['日付'], subset['P_L'].cumsum(), label=f'M{idx} Cumulative P&L')

plt.title(f'Cumulative P&L Simulation ({N_DAYS}d ahead prediction)', fontsize=15)
plt.ylabel('Accumulated Yield Change (%)')
plt.legend()
plt.show()

## 3. 運用効率評価 (シャープレシオ)

In [ ]:
metrics = []
for idx in range(1, 9):
    subset = test_df[test_df['Meeting_Index'] == idx]
    mean_ret = subset['P_L'].mean()
    std_ret = subset['P_L'].std()
    sharpe = (mean_ret / std_ret) * np.sqrt(250 / N_DAYS) if std_ret > 0 else 0
    metrics.append({'Meeting': f'M{idx}', 'Sharpe_Ratio': sharpe, 'Avg_Ret': mean_ret, 'Total_Ret': subset['P_L'].sum()})

display(pd.DataFrame(metrics))

## 4. 信頼度分析 (予測変動幅 vs 的中率)

In [ ]:
test_df['Pred_Move_Abs'] = test_df['Pred_Move'].abs()
test_df['Is_Hit'] = (np.sign(test_df['Pred_Move']) == np.sign(test_df['Actual_Move'])).astype(int)

test_df['Confidence_Bin'] = pd.qcut(test_df['Pred_Move_Abs'], 5, labels=['Low', 'Mid-Low', 'Mid', 'High', 'Extreme'])
confidence_acc = test_df.groupby('Confidence_Bin', observed=True)['Is_Hit'].mean()

plt.figure(figsize=(10, 6))
confidence_acc.plot(kind='bar', color='skyblue')
plt.axhline(0.5, color='red', linestyle='--')
plt.title('Prediction Confidence (Abs Pred Move) vs Accuracy')
plt.ylabel('Directional Accuracy')
plt.ylim(0, 1)
plt.show()